In [ ]:
# ================== Crowd Counting Baseline (ResNet18) ==================
import os, json, random
from glob import glob
from PIL import Image
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")
assert os.path.exists(TRAIN_IMG_DIR) and os.path.exists(TRAIN_LBL_DIR) and os.path.exists(TEST_IMG_DIR)

# ---- Repro ----
random.seed(42)
torch.manual_seed(42)

label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))
img_path_by_name = {
    os.path.basename(lp).replace(".json", ".jpg"): os.path.join(TRAIN_IMG_DIR, os.path.basename(lp).replace(".json", ".jpg"))
    for lp in label_paths
}

# ---- Dataset ----
class CountDataset(Dataset):
    def __init__(self, label_paths, transform):
        self.label_paths = label_paths
        self.transform = transform

    def __len__(self): return len(self.label_paths)

    def __getitem__(self, idx):
        jpath = self.label_paths[idx]
        with open(jpath, "r") as f:
            data = json.load(f)
        img_id = data["img_id"]      # e.g. "1.jpg"
        count  = float(data["human_num"])
        img = Image.open(img_path_by_name[img_id]).convert("RGB")
        img = self.transform(img)
        y = torch.tensor([count], dtype=torch.float32)
        return img, y, img_id

# ---- Transforms ----
train_tf = T.Compose([
    T.Resize((384,384)),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(0.2,0.2,0.2,0.1),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
valid_tf = T.Compose([
    T.Resize((384,384)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ---- Split ----
random.shuffle(label_paths)
split = int(0.9*len(label_paths))
train_ds = CountDataset(label_paths[:split], transform=train_tf)
valid_ds = CountDataset(label_paths[split:], transform=valid_tf)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ---- Model ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m.fc = nn.Sequential(
    nn.Linear(m.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 1)
)
m = m.to(device)

crit = nn.SmoothL1Loss()
opt  = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)

def run_epoch(dl, train=True):
    m.train(train)
    total, n = 0.0, 0
    for x,y,_ in dl:
        x,y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            pred = m(x)
            loss = crit(pred, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*x.size(0); n += x.size(0)
    return total/n

best = 1e9
for epoch in range(20):  
    tr = run_epoch(train_dl, True)
    va = run_epoch(valid_dl, False)
    if va < best:
        best = va
        torch.save(m.state_dict(), "/kaggle/working/best_resnet18.pt")
    print(f"Epoch {epoch+1}: train {tr:.4f} | valid {va:.4f}")

# ---- Inference ----
m.load_state_dict(torch.load("/kaggle/working/best_resnet18.pt", map_location=device))
m.eval()

test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows = []
with torch.no_grad():
    for p in test_imgs:
        img_id = os.path.basename(p)
        img = Image.open(p).convert("RGB")
        x = valid_tf(img).unsqueeze(0).to(device)
        pred = m(x).item()
        rows.append([img_id, int(max(0, round(pred)))])  # non-negative int

sub = pd.DataFrame(rows, columns=["image_id","predicted_count"])

order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub, on="image_id", how="left").fillna(0).astype({"predicted_count": int})

out_path = "/kaggle/working/submission_resnet18.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
import os, json, math, random, gc
from glob import glob
from PIL import Image
import numpy as np
import pandas as pd
import cv2

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import os, multiprocessing as mp
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import cv2, torch
cv2.setNumThreads(0)
torch.set_num_threads(1)
try:
    mp.set_start_method("fork", force=True)  
except RuntimeError:
    pass

# ------------------------ Config ------------------------
COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")

assert os.path.exists(TRAIN_IMG_DIR), TRAIN_IMG_DIR
assert os.path.exists(TRAIN_LBL_DIR), TRAIN_LBL_DIR
assert os.path.exists(TEST_IMG_DIR), TEST_IMG_DIR
assert os.path.exists(SAMPLE_CSV), SAMPLE_CSV

SEED = 42
IMG_SIZE = 512              
BATCH_SIZE = 4              
EPOCHS = 30                 
SIGMA = 8                   
LAM_COUNT = 0.1             
LR = 1e-4
WD = 1e-4
WARMUP_STEPS = 200

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (torch.cuda.is_available() and device.type == "cuda")

# ------------------------ Helpers ------------------------
def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def make_density_map(h, w, points, sigma=8):
    """Fixed-σ gaussian KDE density map."""
    dm = np.zeros((h, w), dtype=np.float32)
    if not points:
        return dm
    for x, y in points:
        x = int(np.clip(x, 0, w-1))
        y = int(np.clip(y, 0, h-1))
        dm[y, x] += 1.0
    k = int(6 * sigma + 1)
    dm = cv2.GaussianBlur(dm, (k, k), sigma, borderType=cv2.BORDER_REFLECT)
    return dm

label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))
img_path_by_name = {}
for lp in label_paths:
    name = os.path.basename(lp).replace(".json", ".jpg")
    ipath = os.path.join(TRAIN_IMG_DIR, name)
    if not os.path.exists(ipath):
        alt = glob(os.path.join(TRAIN_IMG_DIR, os.path.basename(lp).replace(".json", ".*")))
        if alt:
            ipath = alt[0]
    img_path_by_name[os.path.basename(ipath)] = ipath  # key: actual basename available

# ------------------------ Dataset ------------------------
class DensityDataset(Dataset):
    def __init__(self, label_paths, out_size=(IMG_SIZE, IMG_SIZE), aug=True):
        self.label_paths = label_paths
        self.out_w, self.out_h = out_size
        self.aug = aug
        self.norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

    def _normalize_points(self, pts):
        """Normalize to [[float(x), float(y)], ...]. Accept dict/list/tuple/strings."""
        norm = []
        if not pts:
            return norm
        for p in pts:
            try:
                if isinstance(p, dict):
                    x = p.get("x", p.get("X", None))
                    y = p.get("y", p.get("Y", None))
                elif isinstance(p, (list, tuple)) and len(p) >= 2:
                    x, y = p[0], p[1]
                else:
                    continue
                x = float(x); y = float(y)
                norm.append([x, y])
            except Exception:
                continue
        return norm

    def __len__(self): 
        return len(self.label_paths)

    def __getitem__(self, idx):
        jpath = self.label_paths[idx]
        data = load_json(jpath)
        if "img_id" not in data:
            raise ValueError(f"'img_id' missing in {jpath}")
        img_id = data["img_id"]

        pts = self._normalize_points(data.get("points", []))

        ipath_direct = os.path.join(TRAIN_IMG_DIR, img_id)
        if os.path.exists(ipath_direct):
            img_path = ipath_direct
        else:
            if img_path is None:
                stem = os.path.splitext(img_id)[0]
                cand = glob(os.path.join(TRAIN_IMG_DIR, f"{stem}.*"))
                if not cand:
                    raise FileNotFoundError(f"Image for {img_id} not found")
                img_path = cand[0]

        img = Image.open(img_path).convert("RGB")
        w0, h0 = img.size

        img = img.resize((self.out_w, self.out_h), Image.BICUBIC)
        sx, sy = self.out_w / max(1, w0), self.out_h / max(1, h0)

        pts_scaled = [[px * sx, py * sy] for (px, py) in pts]

        if self.aug and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            pts_scaled = [[self.out_w - px, py] for (px, py) in pts_scaled]

        dens = make_density_map(self.out_h, self.out_w, pts_scaled, sigma=SIGMA)

        img_t  = T.ToTensor()(img)
        img_t  = self.norm(img_t)
        dens_t = torch.tensor(dens, dtype=torch.float32).unsqueeze(0)  # [1,H,W]
        count  = torch.tensor([float(len(pts))], dtype=torch.float32)  # GT count (for metrics)
        return img_t, dens_t, count, os.path.basename(img_path)

# ------------------------ Model ------------------------
class CSRNetLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.front = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64,64,3,padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 1/2
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128,128,3,padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 1/4
            nn.Conv2d(128,256,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256,256,3,padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 1/8
            nn.Conv2d(256,512,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=1), nn.ReLU(inplace=True),
        )
        self.back = nn.Sequential(
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,512,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512,256,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256,128,3,padding=2,dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128,64, 3,padding=2,dilation=2), nn.ReLU(inplace=True),
        )
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        x = self.front(x)
        x = self.back(x)
        x = self.out(x)
        return F.relu(x)  

# ------------------------ Data ------------------------
random.shuffle(label_paths)
split = int(0.9 * len(label_paths))
train_ds = DensityDataset(label_paths[:split], out_size=(IMG_SIZE, IMG_SIZE), aug=True)
valid_ds = DensityDataset(label_paths[split:],  out_size=(IMG_SIZE, IMG_SIZE), aug=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ------------------------ Optim, Sched, AMP, EMA ------------------------
net = CSRNetLite().to(device)
loss_mse = nn.MSELoss()
opt  = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD)

total_steps = max(1, EPOCHS * len(train_dl))
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = p.data.clone()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}
ema = EMA(net, decay=0.999)

# ------------------------ Train / Validate with Early Stopping ------------------------
def validate(dl):
    net.eval()
    mae, n = 0.0, 0
    with torch.no_grad():
        for img, dens, count, _ in dl:
            img = img.to(device); dens = dens.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = net(img)
            # match shapes if needed
            if pred.shape[-2:] != dens.shape[-2:]:
                dens_resized = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
            else:
                dens_resized = dens
            pred_count = pred.sum(dim=[1,2,3])
            true_count = dens_resized.sum(dim=[1,2,3])
            mae += (pred_count - true_count).abs().sum().item()
            n   += img.size(0)
    return mae / max(1, n)

best_mae = float("inf")
global_step = 0

PATIENCE = 5            
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    net.train()
    running = 0.0

    for img, dens, count, _ in train_dl:
        img = img.to(device); dens = dens.to(device)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            pred = net(img)
            if pred.shape[-2:] != dens.shape[-2:]:
                dens_resized = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
            else:
                dens_resized = dens
            loss_den = loss_mse(pred, dens_resized)
            loss_cnt = (pred.sum(dim=[1,2,3]) - dens_resized.sum(dim=[1,2,3])).abs().mean()
            loss = loss_den + LAM_COUNT * loss_cnt

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        ema.update(net)
        sched.step()
        global_step += 1
        running += loss.item() * img.size(0)

    # ---- EMA eval ----
    ema.apply_to(net)
    val_mae = validate(valid_dl)
    ema.restore(net)

    train_loss = running / max(1, len(train_ds))

    improved = val_mae < best_mae - 1e-6
    if improved:
        best_mae = val_mae
        no_improve = 0
        torch.save(net.state_dict(), "/kaggle/working/best_csrnet.pt")
        tag = " (improved ✅)"
    else:
        no_improve += 1
        tag = f" (no improve {no_improve}/{PATIENCE})"

    print(f"Epoch {epoch}: train_loss {train_loss:.4f} | val_MAE {val_mae:.2f} | best {best_mae:.2f}{tag}")

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} — best val_MAE {best_mae:.2f}")
        break

# ------------------------ Inference (multi-scale TTA) ------------------------
def predict_count(img_path, scales=(0.75, 1.0, 1.25)):
    img0 = Image.open(img_path).convert("RGB")
    counts = []
    with torch.no_grad():
        for s in scales:
            w = int(IMG_SIZE * s); h = int(IMG_SIZE * s)
            img = img0.resize((w, h), Image.BICUBIC)
            x = T.ToTensor()(img)
            x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
            x = x.unsqueeze(0).to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                y = net(x)
            counts.append(y.sum().item())
    return float(np.mean(counts)) if counts else 0.0

# load best weights
net.load_state_dict(torch.load("/kaggle/working/best_csrnet.pt", map_location=device))
net.eval()

test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows = []
for p in test_imgs:
    img_id = os.path.basename(p)
    c = predict_count(p, scales=(0.75, 1.0, 1.25))
    rows.append([img_id, int(max(0, round(c)))])  # clamp to non-negative int

sub = pd.DataFrame(rows, columns=["image_id", "predicted_count"])
order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub, on="image_id", how="left").fillna(0).astype({"predicted_count": int})

out_path = "/kaggle/working/submission_csrnet.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
import os, json, math, random, warnings
from glob import glob
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# ------------------------ Paths ------------------------
COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")
assert os.path.exists(TRAIN_IMG_DIR) and os.path.exists(TRAIN_LBL_DIR) and os.path.exists(TEST_IMG_DIR) and os.path.exists(SAMPLE_CSV)

# ------------------------ Repro ------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (torch.cuda.is_available() and device.type == "cuda")

# ------------------------ Hyperparams ------------------------
IMG_SIZE      = 512
BATCH_SIZE    = 16
EPOCHS        = 30            
PATIENCE      = 5
LR            = 1e-4
WD            = 1e-4
WARMUP_STEPS  = 200
FOLDS         = 3             
TTA_SCALES    = (1.0, 1.25)   

# ------------------------ Data listing ------------------------
label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))
def read_ann(p):
    with open(p, "r") as f: d = json.load(f)
    return d["img_id"], float(d["human_num"])
meta = [read_ann(p) for p in label_paths]
df = pd.DataFrame(meta, columns=["image_id","count"])
df["img_path"] = df["image_id"].apply(lambda n: os.path.join(TRAIN_IMG_DIR, n))
# Stratify by binned counts (handles class imbalance)
df["bin"] = pd.qcut(df["count"], q=min(10, len(df)//100+2), duplicates="drop", labels=False)

# ------------------------ Dataset ------------------------
class CountDataset(Dataset):
    def __init__(self, df, augment=True):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

        if augment:
            self.tf = T.Compose([
                T.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
                T.RandomHorizontalFlip(0.5),
                T.ColorJitter(0.2, 0.2, 0.2, 0.1),
                T.ToTensor(),
                self.norm,
            ])
        else:
            self.tf = T.Compose([
                T.Resize((IMG_SIZE, IMG_SIZE)),
                T.ToTensor(),
                self.norm,
            ])

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.tf(img)
        y = torch.tensor([math.log1p(row["count"])], dtype=torch.float32)  # log1p target
        return x, y, row["image_id"]

# ------------------------ Model ------------------------
class ResNetReg(nn.Module):
    def __init__(self, backbone="resnet50", pretrained=True, drop=0.25):
        super().__init__()
        if backbone == "resnet18":
            m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            in_f = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m
        else:
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            in_f = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m
        self.head = nn.Sequential(
            nn.Linear(in_f, 512), nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        f = self.backbone(x)
        out = self.head(f)
        return out  

# ------------------------ Train utils ------------------------
def make_loaders(train_idx, valid_idx):
    dtrain = df.iloc[train_idx]; dvalid = df.iloc[valid_idx]
    ds_tr = CountDataset(dtrain, augment=True)
    ds_va = CountDataset(dvalid, augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return dl_tr, dl_va, dvalid

def cosine_warmup(total_steps):
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return lr_lambda

def train_one_fold(fold, train_idx, valid_idx, out_dir="/kaggle/working"):
    dl_tr, dl_va, dvalid = make_loaders(train_idx, valid_idx)

    model = ResNetReg("resnet50").to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    total_steps = max(1, EPOCHS * len(dl_tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, cosine_warmup(total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    crit   = nn.MSELoss()  

    best = float("inf"); no_improve = 0
    best_path = os.path.join(out_dir, f"resnet50_fold{fold}.pt")

    def validate():
        model.eval()
        mae, n = 0.0, 0
        with torch.no_grad():
            for x, y, _ in dl_va:
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    pred = model(x)                
                    pred_cnt = torch.expm1(pred)  
                    true_cnt = torch.expm1(y)
                mae += (pred_cnt - true_cnt).abs().sum().item()
                n   += x.size(0)
        return mae / max(1, n)

    for epoch in range(1, EPOCHS+1):
        model.train()
        running = 0.0; seen = 0
        for x, y, _ in dl_tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(x)         # log1p
                loss = crit(pred, y)    # MSE on log1p
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item() * x.size(0); seen += x.size(0)

        val_mae = validate()
        tr_loss = running / max(1, seen)

        improved = val_mae < best - 1e-6
        if improved:
            best = val_mae; no_improve = 0
            torch.save(model.state_dict(), best_path)
            tag = " (improved ✅)"
        else:
            no_improve += 1; tag = f" (no improve {no_improve}/{PATIENCE})"

        print(f"[Fold {fold}] Epoch {epoch}: train_loss {tr_loss:.4f} | val_MAE {val_mae:.3f} | best {best:.3f}{tag}")
        if no_improve >= PATIENCE:
            print(f"[Fold {fold}] Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    return model, best_path

# ------------------------ K-Fold training ------------------------
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_models = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["bin"]), start=1):
    model, path = train_one_fold(fold, tr_idx, va_idx)
    fold_models.append(path)

# ------------------------ Inference with TTA ------------------------
def predict_count(models_paths, img_path):
    img0 = Image.open(img_path).convert("RGB")
    preds = []
    for scale in TTA_SCALES:
        if scale == 1.0:
            img = img0.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            flips = [False, True]
        else:
            sz = int(IMG_SIZE * scale)
            img = img0.resize((sz, sz), Image.BICUBIC)
            flips = [False]  

        for flip in flips:
            im = img.transpose(Image.FLIP_LEFT_RIGHT) if flip else img
            x = T.ToTensor()(im)
            x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
            x = x.unsqueeze(0).to(device)

            fold_outs = []
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
                for mp in models_paths:
                    m = ResNetReg("resnet50"); m.load_state_dict(torch.load(mp, map_location=device)); m.to(device); m.eval()
                    y = m(x)  
                    fold_outs.append(torch.expm1(y).item())
            preds.append(np.mean(fold_outs))
    return float(np.mean(preds)) if preds else 0.0

_loaded = []
for mp in fold_models:
    m = ResNetReg("resnet50"); m.load_state_dict(torch.load(mp, map_location=device)); m.to(device); m.eval()
    _loaded.append(m)
def predict_count_cached(models_mem, img_path):
    img0 = Image.open(img_path).convert("RGB")
    preds = []
    for scale in TTA_SCALES:
        sz = int(IMG_SIZE * scale)
        img = img0.resize((sz, sz), Image.BICUBIC)
        for flip in ([False, True] if scale == 1.0 else [False]):
            im = img.transpose(Image.FLIP_LEFT_RIGHT) if flip else img
            x = T.ToTensor()(im)
            x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
            x = x.unsqueeze(0).to(device)
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
                outs = [torch.expm1(m(x)).item() for m in models_mem]
            preds.append(np.mean(outs))
    return float(np.mean(preds)) if preds else 0.0

# ------------------------ Build submission ------------------------
test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows = []
for p in test_imgs:
    img_id = os.path.basename(p)
    c = predict_count_cached(_loaded, p)
    rows.append([img_id, int(max(0, round(c)))])

sub = pd.DataFrame(rows, columns=["image_id","predicted_count"])
order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub, on="image_id", how="left").fillna(0).astype({"predicted_count": int})

out_path = "/kaggle/working/submission_resnet50_k3.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path, " | rows:", len(sub))

In [1]:
!pip install timm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.0 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os, json, math, random, warnings
from glob import glob
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm  # Swin backbone

# ---------------- Paths ----------------
COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")
assert os.path.exists(TRAIN_IMG_DIR) and os.path.exists(TRAIN_LBL_DIR) and os.path.exists(TEST_IMG_DIR) and os.path.exists(SAMPLE_CSV)

# ---------------- Config ----------------
SEED = 42
IMG_SIZE      = 224         # Swin-Tiny native size
BATCH_SIZE    = 8
EPOCHS        = 30
PATIENCE      = 5
LR            = 3e-5        # lower LR for transformers
WD            = 1e-4
WARMUP_STEPS  = 200
FOLDS         = 3
TTA_SCALES    = (1.0,)      # flip-only TTA at native size (avoid 280/other sizes)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (torch.cuda.is_available() and device.type == "cuda")

# ---------------- Dataset prep ----------------
label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))
def read_ann(p):
    with open(p, "r") as f: d = json.load(f)
    return d["img_id"], float(d["human_num"])
meta = [read_ann(p) for p in label_paths]
df = pd.DataFrame(meta, columns=["image_id","count"])
df["img_path"] = df["image_id"].apply(lambda n: os.path.join(TRAIN_IMG_DIR, n))
df["bin"] = pd.qcut(df["count"], q=min(10, len(df)//100+2), duplicates="drop", labels=False)

class CountDataset(Dataset):
    def __init__(self, df, augment=True):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        if augment:
            self.tf = T.Compose([
                T.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
                T.RandomHorizontalFlip(0.5),
                T.ColorJitter(0.2, 0.2, 0.2, 0.1),
                T.ToTensor(), self.norm
            ])
        else:
            self.tf = T.Compose([
                T.Resize((IMG_SIZE, IMG_SIZE)),
                T.ToTensor(), self.norm
            ])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.tf(img)
        y = torch.tensor([math.log1p(row["count"])], dtype=torch.float32)  # log1p target
        return x, y, row["image_id"]

# ---------------- Swin model ----------------
class SwinReg(nn.Module):
    def __init__(self, model_name="swin_tiny_patch4_window7_224", drop=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,      # feature extractor
            global_pool="avg"   # avg pool to feature vector
        )
        in_f = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(in_f, 512), nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(512, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)  # predicts log1p(count)

# ---------------- Training utils ----------------
def make_loaders(train_idx, valid_idx):
    dtrain = df.iloc[train_idx]; dvalid = df.iloc[valid_idx]
    ds_tr = CountDataset(dtrain, augment=True)
    ds_va = CountDataset(dvalid, augment=False)
    return (
        DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0),
        DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
        dvalid
    )

def cosine_warmup(total_steps):
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return lr_lambda

def train_one_fold(fold, train_idx, valid_idx, out_dir="/kaggle/working"):
    dl_tr, dl_va, dvalid = make_loaders(train_idx, valid_idx)
    model = SwinReg().to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    total_steps = max(1, EPOCHS * len(dl_tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, cosine_warmup(total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    crit   = nn.MSELoss()  # on log1p(count)

    best = float("inf"); no_improve = 0
    best_path = os.path.join(out_dir, f"swin_fold{fold}.pt")

    def validate():
        model.eval()
        mae, n = 0.0, 0
        with torch.no_grad():
            for x, y, _ in dl_va:
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    pred = model(x)                # log1p
                    pred_cnt = torch.expm1(pred)  # back to count
                    true_cnt = torch.expm1(y)
                mae += (pred_cnt - true_cnt).abs().sum().item()
                n   += x.size(0)
        return mae / max(1, n)

    for epoch in range(1, EPOCHS+1):
        model.train(); running = 0.0; seen = 0
        for x, y, _ in dl_tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(x); loss = crit(pred, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()*x.size(0); seen += x.size(0)

        val_mae = validate(); tr_loss = running/max(1,seen)
        improved = val_mae < best - 1e-6
        if improved:
            best = val_mae; no_improve = 0
            torch.save(model.state_dict(), best_path)
            tag = " (improved ✅)"
        else:
            no_improve += 1; tag = f" (no improve {no_improve}/{PATIENCE})"
        print(f"[Fold {fold}] Epoch {epoch}: train_loss {tr_loss:.4f} | val_MAE {val_mae:.3f} | best {best:.3f}{tag}")
        if no_improve >= PATIENCE:
            print(f"[Fold {fold}] Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    return model, best_path

# ---------------- K-Fold training ----------------
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_models = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["bin"]), start=1):
    model, path = train_one_fold(fold, tr_idx, va_idx)
    fold_models.append(path)

# ---------------- Inference with flip-only TTA (size 224) ----------------
_loaded = []
for mp in fold_models:
    m = SwinReg(); m.load_state_dict(torch.load(mp, map_location=device)); m.to(device); m.eval()
    _loaded.append(m)

def predict_count_cached(models_mem, img_path):
    img0 = Image.open(img_path).convert("RGB")
    preds = []
    for scale in TTA_SCALES:           # only (1.0,)
        sz = int(IMG_SIZE * float(scale))  # = 224
        img = img0.resize((sz, sz), Image.BICUBIC)
        for flip in [False, True]:     # flip-only TTA at native size
            im = img.transpose(Image.FLIP_LEFT_RIGHT) if flip else img
            x = T.ToTensor()(im); x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
            x = x.unsqueeze(0).to(device)
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp):
                outs = [torch.expm1(m(x)).item() for m in models_mem]
            preds.append(np.mean(outs))
    return float(np.mean(preds)) if preds else 0.0

# ---------------- Submission ----------------
test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows = []
for p in test_imgs:
    img_id = os.path.basename(p)
    c = predict_count_cached(_loaded, p)
    rows.append([img_id, int(max(0, round(c)))])

sub = pd.DataFrame(rows, columns=["image_id","predicted_count"])
order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub, on="image_id", how="left").fillna(0).astype({"predicted_count": int})

out_path = "/kaggle/working/submission_swint_k3.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(sub))

In [ ]:
# ================= Crowd Counting — ResNet18 Density (Adaptive-σ, 3-Fold, 512px) =================
# - Builds geometry-adaptive (KNN) Gaussian density maps from JSON points
# - ResNet18 encoder (1/8) + dilated head w/ Softplus
# - MSE+L1 density loss + small count consistency loss
# - Cosine LR w/ warmup, AMP, EMA, Early Stopping
# - 3-fold CV with per-fold linear calibration (y_true ≈ a*y_pred + b)
# - TTA (scales 0.75/1.0/1.25 + flips)
# Output: /kaggle/working/submission_resnet18_density_k3.csv

import os, json, math, random, warnings
from glob import glob
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

# ---------------------- Paths ----------------------
COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")
assert all(os.path.exists(p) for p in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, TEST_IMG_DIR, SAMPLE_CSV])

# ---------------------- Config ----------------------
SEED = 42
IMG_SIZE = 512          # go 512 for tiny heads; drop batch size if OOM
OUT_STRIDE = 8          # model output stride (H/8, W/8)
BATCH_SIZE = 6          # 6–8 for 512px; bump if VRAM allows
EPOCHS = 40
PATIENCE = 6
LR = 1.5e-4             # slightly higher to avoid flat starts; lower to 1e-4 if unstable
WD = 1e-4
WARMUP_STEPS = 200
LAM_COUNT = 0.1         # weight for count loss
FOLDS = 3
TTA_SCALES = (0.75, 1.0, 1.25)  # multi-scale
TTA_FLIP = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (torch.cuda.is_available() and device.type == "cuda")

# ---------------------- Dataframe (for stratified folds) ----------------------
label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))

def read_meta(p):
    with open(p, "r") as f:
        d = json.load(f)
    return d["img_id"], int(d.get("human_num", len(d.get("points", [])) or 0))

meta = [read_meta(p) for p in label_paths]
df = pd.DataFrame(meta, columns=["image_id", "count"])
df["img_path"] = df["image_id"].apply(lambda n: os.path.join(TRAIN_IMG_DIR, n))
# robust image path (handle extension mismatches)
for i, r in df.iterrows():
    if not os.path.exists(r["img_path"]):
        stem = os.path.splitext(r["image_id"])[0]
        alts = glob(os.path.join(TRAIN_IMG_DIR, f"{stem}.*"))
        if alts: df.at[i, "img_path"] = alts[0]
df["bin"] = pd.qcut(df["count"], q=min(10, max(2, len(df)//100+2)), duplicates="drop", labels=False)

# ---------------------- Utils ----------------------
def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def normalize_points(pts):
    out=[]
    if not pts: return out
    for p in pts:
        try:
            if isinstance(p, dict):
                x = p.get("x", p.get("X", None)); y = p.get("y", p.get("Y", None))
            else:
                x, y = p[0], p[1]
            out.append([float(x), float(y)])
        except Exception:
            pass
    return out

def make_density_map_adaptive(h, w, points, k=3, beta=0.3, min_sigma=1.5, max_sigma=10.0):
    """
    Geometry-adaptive kernels (Zhang et al.):
    sigma_i = beta * mean distance to k nearest neighbors (in output-grid coords).
    """
    dm = np.zeros((h, w), dtype=np.float32)
    if not points:
        return dm

    P = np.array(points, dtype=np.float32)
    # initial impulses (not strictly necessary before rendering Gaussians, but OK)
    # for xg, yg in P:
    #     xi = int(np.clip(xg, 0, w-1)); yi = int(np.clip(yg, 0, h-1))
    #     dm[yi, xi] += 1.0

    # estimate sigmas
    if len(P) >= 2:
        nb = min(k+1, len(P))
        nn = NearestNeighbors(n_neighbors=nb, algorithm="auto").fit(P)
        dists, _ = nn.kneighbors(P)  # includes self at 0
        if dists.shape[1] > 1:
            mean_k = dists[:, 1:1+k].mean(axis=1)
        else:
            mean_k = np.full(len(P), 1.0, dtype=np.float32)
        sigmas = np.clip(beta * mean_k, min_sigma, max_sigma)
    else:
        sigmas = np.full(len(P), 4.0, dtype=np.float32)

    # render per-point Gaussian; normalize by 2πσ^2 so sums approximate counts
    for (xg, yg), s in zip(P, sigmas):
        xi = int(np.clip(xg, 0, w-1)); yi = int(np.clip(yg, 0, h-1))
        r = int(max(1, 3*s))
        x0, x1 = max(0, xi-r), min(w, xi+r+1)
        y0, y1 = max(0, yi-r), min(h, yi+r+1)
        yy, xx = np.ogrid[y0:y1, x0:x1]
        g = np.exp(-((xx-xi)**2 + (yy-yi)**2) / (2*s*s)) / (2*np.pi*s*s)
        dm[y0:y1, x0:x1] += g.astype(np.float32)
    return dm

# ---------------------- Dataset ----------------------
class DensityDataset(Dataset):
    def __init__(self, idxs, img_size=IMG_SIZE, out_stride=OUT_STRIDE, augment=True):
        self.df = df.iloc[idxs].reset_index(drop=True)
        self.img_size = img_size
        self.out_stride = out_stride
        self.dh = img_size // out_stride
        self.dw = img_size // out_stride
        self.augment = augment
        self.norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

        if augment:
            self.geom_aug = True
            self.color_tf = T.ColorJitter(0.2,0.2,0.2,0.1)
        else:
            self.geom_aug = False
            self.color_tf = None

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        jpath = os.path.join(TRAIN_LBL_DIR, os.path.splitext(os.path.basename(row["img_path"]))[0] + ".json")
        data = load_json(jpath)
        pts = normalize_points(data.get("points", []))

        img = Image.open(row["img_path"]).convert("RGB")
        w0, h0 = img.size

        # resize to fixed size, scale points the same
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        sx, sy = self.img_size / max(1,w0), self.img_size / max(1,h0)
        pts_img = [[px*sx, py*sy] for (px,py) in pts]

        # light geometric aug: horizontal flip
        if self.geom_aug and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            pts_img = [[self.img_size - px, py] for (px,py) in pts_img]

        # color aug
        if self.color_tf is not None:
            img = self.color_tf(img)

        # build density on output grid
        pts_grid = [[px/self.out_stride, py/self.out_stride] for (px,py) in pts_img]
        dens = make_density_map_adaptive(self.dh, self.dw, pts_grid, k=3, beta=0.3)

        x = T.ToTensor()(img); x = self.norm(x)
        dens_t = torch.tensor(dens, dtype=torch.float32).unsqueeze(0)
        cnt = torch.tensor([float(len(pts))], dtype=torch.float32)
        return x, dens_t, cnt, os.path.basename(row["img_path"])

# ---------------------- Model ----------------------
class ResNet18Density(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(m.conv1, m.bn1, nn.ReLU(inplace=True), m.maxpool)  # stride 4
        self.layer1 = m.layer1  # stride 4
        self.layer2 = m.layer2  # stride 8 → [B,128,H/8,W/8]
        self.head = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128,  64, 3, padding=1),             nn.ReLU(inplace=True),
            nn.Conv2d(64,    1, 1)
        )
        self.out_act = nn.Softplus(beta=1.0)
        last = self.head[-1]
        if isinstance(last, nn.Conv2d) and last.bias is not None:
            nn.init.constant_(last.bias, 0.01)

    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x)
        y = self.head(x)
        return self.out_act(y)  # [B,1,H/8,W/8] ≥ 0

# ---------------------- Train utils ----------------------
def cosine_warmup(total_steps):
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return lr_lambda

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.clone() for n,p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)
    def apply_to(self, model):
        self.backup = {}
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}

def validate(model, dl, amp=True):
    model.eval(); mae, n = 0.0, 0
    with torch.no_grad():
        for x, dens, _c, _ in dl:
            x, dens = x.to(device), dens.to(device)
            with torch.amp.autocast("cuda", enabled=amp):
                pred = model(x)
                if pred.shape[-2:] != dens.shape[-2:]:
                    dens = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
            mae += (pred.sum([1,2,3]) - dens.sum([1,2,3])).abs().sum().item()
            n += x.size(0)
    return mae / max(1, n)

def train_one_fold(fold, idx_tr, idx_va, out_dir="/kaggle/working"):
    # loaders
    dl_tr = DataLoader(DensityDataset(idx_tr, augment=True),  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    dl_va = DataLoader(DensityDataset(idx_va, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # model/opt
    net = ResNet18Density(pretrained=True).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD)
    loss_mse = nn.MSELoss()

    total_steps = max(1, EPOCHS * len(dl_tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, cosine_warmup(total_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ema = EMA(net)

    best_mae = float("inf"); no_improve = 0
    best_path = os.path.join(out_dir, f"r18_density_fold{fold}.pt")

    for epoch in range(1, EPOCHS+1):
        net.train(); run_loss, seen = 0.0, 0
        for x, dens, _c, _ in dl_tr:
            x, dens = x.to(device), dens.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = net(x)
                if pred.shape[-2:] != dens.shape[-2:]:
                    dens = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
                # hybrid density loss (MSE + L1) + count consistency
                loss_den = 0.5*loss_mse(pred, dens) + 0.5*F.l1_loss(pred, dens)
                loss_cnt = (pred.sum([1,2,3]) - dens.sum([1,2,3])).abs().mean()
                loss = loss_den + LAM_COUNT * loss_cnt
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(net)
            run_loss += loss.item() * x.size(0); seen += x.size(0)

        # EMA validation
        ema.apply_to(net)
        val_mae = validate(net, dl_va, amp=use_amp)
        ema.restore(net)

        tr_loss = run_loss / max(1, seen)
        if val_mae < best_mae - 1e-6:
            best_mae = val_mae; no_improve = 0
            torch.save(net.state_dict(), best_path)
            tag = " (improved ✅)"
        else:
            no_improve += 1
            tag = f" (no improve {no_improve}/{PATIENCE})"
        print(f"[Fold {fold}] Epoch {epoch}: train_loss {tr_loss:.4f} | val_MAE {val_mae:.3f} | best {best_mae:.3f}{tag}")
        if no_improve >= PATIENCE:
            print(f"[Fold {fold}] Early stopping at epoch {epoch}"); break

    # reload best
    net.load_state_dict(torch.load(best_path, map_location=device))
    net.eval()

    # ----- Per-fold linear calibration on its validation set -----
    preds, trues = [], []
    with torch.no_grad():
        for x, dens, _c, _ in dl_va:
            x, dens = x.to(device), dens.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                p = net(x).sum(dim=[1,2,3]).detach().cpu().numpy()
            t = dens.sum(dim=[1,2,3]).detach().cpu().numpy()
            preds.extend(p.tolist()); trues.extend(t.tolist())
    if len(preds) >= 2:
        a, b = np.polyfit(np.array(preds), np.array(trues), deg=1)
    else:
        a, b = 1.0, 0.0
    print(f"[Fold {fold}] Calibration: a={a:.4f}, b={b:.4f}")

    return best_path, (float(a), float(b))

# ---------------------- K-Fold ----------------------
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_paths, fold_calibs = [], []
for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df["bin"]), start=1):
    best_path, calib = train_one_fold(fold, tr_idx, va_idx)
    fold_paths.append(best_path); fold_calibs.append(calib)

# ---------------------- Load models for inference ----------------------
models_mem = []
for mp in fold_paths:
    m = ResNet18Density(pretrained=False).to(device)
    m.load_state_dict(torch.load(mp, map_location=device))
    m.eval()
    models_mem.append(m)

# ---------------------- Inference (TTA + per-fold calibration) ----------------------
def predict_count(img_path, scales=TTA_SCALES, do_flip=TTA_FLIP):
    img0 = Image.open(img_path).convert("RGB")
    # Aggregate per-fold (with each fold's calibration)
    fold_counts = []
    with torch.no_grad():
        for m, (a, b) in zip(models_mem, fold_calibs):
            tta_counts = []
            for s in scales:
                w = int(IMG_SIZE * s); h = int(IMG_SIZE * s)
                resized = img0.resize((w, h), Image.BICUBIC)
                flips = [False, True] if do_flip else [False]
                for flip in flips:
                    im = resized.transpose(Image.FLIP_LEFT_RIGHT) if flip else resized
                    x = T.Resize((IMG_SIZE, IMG_SIZE))(im)
                    x = T.ToTensor()(x); x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
                    x = x.unsqueeze(0).to(device)
                    with torch.amp.autocast("cuda", enabled=use_amp):
                        dens = m(x)            # [1,1,dh,dw]
                        c = dens.sum().item()  # raw count
                    # per-fold calibration
                    c_cal = max(0.0, a * c + b)
                    tta_counts.append(c_cal)
            fold_counts.append(float(np.mean(tta_counts)))
    return float(np.mean(fold_counts)) if fold_counts else 0.0

# ---------------------- Submission ----------------------
test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows = []
for p in test_imgs:
    img_id = os.path.basename(p)
    c = predict_count(p)
    rows.append([img_id, int(max(0, round(c)))])

sub = pd.DataFrame(rows, columns=["image_id","predicted_count"])
order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub, on="image_id", how="left").fillna(0).astype({"predicted_count": int})

out_path = "/kaggle/working/submission_resnet18_density_k3.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(sub))

In [7]:
# ================= Crowd Counting — Swin-Tiny (224) Density Map =================

import os, json, math, random, warnings
from glob import glob
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.neighbors import NearestNeighbors

# ---------------- Paths ----------------
COMP_DIR       = "/kaggle/input/penyisihan-hology-8-0-2025-data-mining"
TRAIN_IMG_DIR  = os.path.join(COMP_DIR, "train", "images")
TRAIN_LBL_DIR  = os.path.join(COMP_DIR, "train", "labels")
TEST_IMG_DIR   = os.path.join(COMP_DIR, "test",  "images")
SAMPLE_CSV     = os.path.join(COMP_DIR, "sample_submission.csv")
assert all(os.path.exists(p) for p in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, TEST_IMG_DIR, SAMPLE_CSV])

# ---------------- Config ----------------
SEED = 42
IMG_SIZE   = 224
OUT_STRIDE = 8
BATCH_SIZE = 16
EPOCHS     = 30
PATIENCE   = 6
LR         = 3e-5
WD         = 1e-4
WARMUP_STEPS = 200
LAM_COUNT  = 0.1
TTA_FLIP   = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["OMP_NUM_THREADS"]="1"; os.environ["MKL_NUM_THREADS"]="1"; torch.set_num_threads(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (torch.cuda.is_available() and device.type == "cuda")

# ---------------- Helpers ----------------
def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def normalize_points(pts):
    out=[]
    if not pts: return out
    for p in pts:
        try:
            if isinstance(p, dict):
                x = p.get("x", p.get("X", None)); y = p.get("y", p.get("Y", None))
            else:
                x, y = p[0], p[1]
            out.append([float(x), float(y)])
        except Exception:
            pass
    return out

def make_density_map_adaptive(h, w, points, k=3, beta=0.3, min_sigma=1.5, max_sigma=10.0):
    dm = np.zeros((h, w), dtype=np.float32)
    if not points: return dm
    P = np.array(points, dtype=np.float32)
    if len(P) >= 2:
        nb = min(k+1, len(P))
        nn = NearestNeighbors(n_neighbors=nb).fit(P)
        dists, _ = nn.kneighbors(P)
        if dists.shape[1] > 1:
            mean_k = dists[:, 1:1+k].mean(axis=1)
        else:
            mean_k = np.full(len(P), 1.0, dtype=np.float32)
        sigmas = np.clip(beta * mean_k, min_sigma, max_sigma)
    else:
        sigmas = np.full(len(P), 4.0, dtype=np.float32)

    for (xg, yg), s in zip(P, sigmas):
        xi = int(np.clip(xg, 0, w-1)); yi = int(np.clip(yg, 0, h-1))
        r = int(max(1, 3*s))
        x0, x1 = max(0, xi-r), min(w, xi+r+1)
        y0, y1 = max(0, yi-r), min(h, yi+r+1)
        yy, xx = np.ogrid[y0:y1, x0:x1]
        g = np.exp(-((xx-xi)**2 + (yy-yi)**2)/(2*s*s)) / (2*np.pi*s*s)
        dm[y0:y1, x0:x1] += g.astype(np.float32)
    return dm

# ---------------- Dataset ----------------
class SwinDensityDataset(Dataset):
    def __init__(self, label_paths, img_size=IMG_SIZE, out_stride=OUT_STRIDE, augment=True):
        self.label_paths = label_paths
        self.img_size = img_size
        self.out_stride = out_stride
        self.dh = img_size // out_stride
        self.dw = img_size // out_stride
        self.augment = augment
        self.norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        if augment:
            self.tf_color = T.ColorJitter(0.2,0.2,0.2,0.1)
        else:
            self.tf_color = None

    def __len__(self): return len(self.label_paths)

    def __getitem__(self, idx):
        jpath = self.label_paths[idx]
        data = load_json(jpath)
        img_id = data["img_id"]
        pts = normalize_points(data.get("points", []))

        ipath = os.path.join(TRAIN_IMG_DIR, img_id)
        if not os.path.exists(ipath):
            stem = os.path.splitext(img_id)[0]
            alts = glob(os.path.join(TRAIN_IMG_DIR, f"{stem}.*"))
            ipath = alts[0]

        img = Image.open(ipath).convert("RGB")
        w0, h0 = img.size
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        sx, sy = self.img_size / max(1,w0), self.img_size / max(1,h0)
        pts_img = [[px*sx, py*sy] for (px,py) in pts]

        if self.augment and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            pts_img = [[self.img_size - px, py] for (px,py) in pts_img]

        if self.tf_color is not None:
            img = self.tf_color(img)

        pts_grid = [[px/self.out_stride, py/self.out_stride] for (px,py) in pts_img]
        dens = make_density_map_adaptive(self.dh, self.dw, pts_grid, k=3, beta=0.3)

        x = T.ToTensor()(img); x = self.norm(x)
        dens_t = torch.tensor(dens, dtype=torch.float32).unsqueeze(0)
        cnt = torch.tensor([float(len(pts))], dtype=torch.float32)
        return x, dens_t, cnt, os.path.basename(ipath)

# ---------------- Swin Density Head ----------------
class SwinTinyDensity(nn.Module):
    def __init__(self, model_name="swin_tiny_patch4_window7_224"):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            features_only=True,
            out_indices=(1,),  # stride-8 feature map
        )
        self.c_out = self.backbone.feature_info.channels()[0]  # expected channels (192 for Swin-T)
        self.head = nn.Sequential(
            nn.Conv2d(self.c_out, 256, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128,  64, 3, padding=1),             nn.ReLU(inplace=True),
            nn.Conv2d(64,    1, 1)
        )
        self.out_act = nn.Softplus(beta=1.0)
        last = self.head[-1]
        if isinstance(last, nn.Conv2d) and last.bias is not None:
            nn.init.constant_(last.bias, 0.01)

    def forward(self, x):
        feats = self.backbone(x)   # list with one tensor
        f = feats[0]               # could be NCHW or NHWC depending on env

        # ---- Ensure NCHW ----
        if f.ndim == 4:
            # If channels are in the last dim (NHWC), permute to NCHW
            if f.shape[-1] == self.c_out and f.shape[1] != self.c_out:
                f = f.permute(0, 3, 1, 2).contiguous()

        y = self.head(f)
        return self.out_act(y)

# ---------------- Split ----------------
label_paths = sorted(glob(os.path.join(TRAIN_LBL_DIR, "*.json")))
random.shuffle(label_paths)
split = int(0.9 * len(label_paths))
train_paths, valid_paths = label_paths[:split], label_paths[split:]

train_ds = SwinDensityDataset(train_paths, augment=True)
valid_ds = SwinDensityDataset(valid_paths, augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ---------------- Train ----------------
net = SwinTinyDensity().to(device)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD)
loss_mse = nn.MSELoss()

total_steps = max(1, EPOCHS * len(train_dl))
def lr_lambda(step):
    if step < WARMUP_STEPS: return step / max(1, WARMUP_STEPS)
    prog = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * prog))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_mae=float("inf"); no_improve=0
for epoch in range(1, EPOCHS+1):
    net.train(); run_loss,seen=0.0,0
    for x, dens, _c, _ in train_dl:
        x,dens = x.to(device), dens.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            pred = net(x)
            if pred.shape[-2:] != dens.shape[-2:]:
                dens = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
            loss_den = 0.5*loss_mse(pred, dens) + 0.5*F.l1_loss(pred, dens)
            loss_cnt = (pred.sum([1,2,3]) - dens.sum([1,2,3])).abs().mean()
            loss = loss_den + LAM_COUNT*loss_cnt
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        run_loss += loss.item()*x.size(0); seen += x.size(0)

    # validation
    net.eval(); mae,n=0.0,0
    with torch.no_grad():
        for x, dens, _c, _ in valid_dl:
            x,dens = x.to(device), dens.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = net(x)
                if pred.shape[-2:] != dens.shape[-2:]:
                    dens = F.interpolate(dens, size=pred.shape[-2:], mode="bilinear", align_corners=False)
            mae += (pred.sum([1,2,3]) - dens.sum([1,2,3])).abs().sum().item()
            n += x.size(0)
    val_mae = mae/max(1,n)
    tr_loss = run_loss/max(1,seen)
    if val_mae < best_mae - 1e-6:
        best_mae, no_improve = val_mae, 0
        torch.save(net.state_dict(), "/kaggle/working/best_swin_tiny_density.pt")
        tag=" (improved ✅)"
    else:
        no_improve += 1; tag=f" (no improve {no_improve}/{PATIENCE})"
    print(f"Epoch {epoch}: train_loss {tr_loss:.4f} | val_MAE {val_mae:.3f} | best {best_mae:.3f}{tag}")
    if no_improve>=PATIENCE:
        print(f"Early stopping at epoch {epoch}"); break

# ---------------- Inference ----------------
net.load_state_dict(torch.load("/kaggle/working/best_swin_tiny_density.pt", map_location=device))
net.eval()

def predict_count(img_path, do_flip=TTA_FLIP):
    img0 = Image.open(img_path).convert("RGB")
    counts = []
    with torch.no_grad():
        for flip in ([False, True] if do_flip else [False]):
            im = img0.transpose(Image.FLIP_LEFT_RIGHT) if flip else img0
            im = im.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            x = T.ToTensor()(im); x = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(x)
            x = x.unsqueeze(0).to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                dens = net(x)
            counts.append(dens.sum().item())
    return float(np.mean(counts)) if counts else 0.0

test_imgs = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
rows=[]
for p in test_imgs:
    img_id = os.path.basename(p)
    c = predict_count(p)
    rows.append([img_id, int(max(0, round(c)))])

sub = pd.DataFrame(rows, columns=["image_id","predicted_count"])
order = pd.read_csv(SAMPLE_CSV)
sub = order[["image_id"]].merge(sub,on="image_id",how="left").fillna(0).astype({"predicted_count":int})

out_path = "/kaggle/working/submission_swin_tiny_density.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(sub))

Epoch 1: train_loss 34.0033 | val_MAE 91.095 | best 91.095 (improved ✅)
Epoch 2: train_loss 9.1676 | val_MAE 54.532 | best 54.532 (improved ✅)
Epoch 3: train_loss 6.7828 | val_MAE 37.761 | best 37.761 (improved ✅)
Epoch 4: train_loss 6.1027 | val_MAE 38.863 | best 37.761 (no improve 1/6)
Epoch 5: train_loss 5.0192 | val_MAE 31.597 | best 31.597 (improved ✅)
Epoch 6: train_loss 4.6227 | val_MAE 25.501 | best 25.501 (improved ✅)
Epoch 7: train_loss 4.2368 | val_MAE 25.516 | best 25.501 (no improve 1/6)
Epoch 8: train_loss 4.0368 | val_MAE 24.523 | best 24.523 (improved ✅)
Epoch 9: train_loss 3.8401 | val_MAE 27.364 | best 24.523 (no improve 1/6)
Epoch 10: train_loss 3.6726 | val_MAE 25.652 | best 24.523 (no improve 2/6)
Epoch 11: train_loss 3.5797 | val_MAE 27.984 | best 24.523 (no improve 3/6)
Epoch 12: train_loss 3.6119 | val_MAE 23.350 | best 23.350 (improved ✅)
Epoch 13: train_loss 3.4603 | val_MAE 32.624 | best 23.350 (no improve 1/6)
Epoch 14: train_loss 3.3158 | val_MAE 25.906 | b